In [ ]:
# 06 - Feature Engineering: Join All Data Sources
# Week 3 - Day 2
# Goal: Merge hourly demand with weather, station metadata, and holidays

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BUCKET = 'bluebikes-demand-predictor-data'
STORAGE_OPTS = {"token": "google_default"}

print('Setup complete!')

Setup complete!


In [2]:
# 1. Hourly demand (from notebook 05)
print('Loading hourly demand...')
demand = pd.read_parquet(
    f'gs://{BUCKET}/processed/features/hourly_demand_by_station.parquet',
    storage_options=STORAGE_OPTS
)
print(f'  Demand: {len(demand):,} rows')

# 2. Weather data
print('Loading weather...')
weather = pd.read_parquet(
    f'gs://{BUCKET}/data/weather/weather_hourly_2023_2024.parquet',
    storage_options=STORAGE_OPTS
)
print(f'  Weather: {len(weather):,} rows')

# 3. Station metadata
print('Loading stations...')
stations = pd.read_parquet(
    f'gs://{BUCKET}/metadata/stations/stations.parquet',
    storage_options=STORAGE_OPTS
)
print(f'  Stations: {len(stations):,} rows')

# 4. Holidays
print('Loading holidays...')
holidays = pd.read_parquet(
    f'gs://{BUCKET}/data/contextual/us_holidays_2023_2024.parquet',
    storage_options=STORAGE_OPTS
)
print(f'  Holidays: {len(holidays):,} rows')

Loading hourly demand...
  Demand: 8,215,056 rows
Loading weather...
  Weather: 15,384 rows
Loading stations...
  Stations: 595 rows
Loading holidays...
  Holidays: 24 rows


In [3]:
print('=== DEMAND ===')
print(demand.dtypes)
print(demand.head(2))

print('\n=== WEATHER ===')
print(weather.columns.tolist())
print(weather.dtypes)
print(weather.head(2))

print('\n=== STATIONS ===')
print(stations.columns.tolist())
print(stations.dtypes)
print(stations.head(2))

print('\n=== HOLIDAYS ===')
print(holidays.columns.tolist())
print(holidays.dtypes)
print(holidays.head(2))

=== DEMAND ===
start_station_id            object
hour                datetime64[ns]
demand_count                 int64
hour_of_day                  int32
date                        object
year                         int32
month                        int32
day_of_week                  int32
is_weekend                   int64
dtype: object
  start_station_id                hour  demand_count  hour_of_day        date  \
0           A32000 2023-04-01 00:00:00             0            0  2023-04-01   
1           A32000 2023-04-01 01:00:00             0            1  2023-04-01   

   year  month  day_of_week  is_weekend  
0  2023      4            5           1  
1  2023      4            5           1  

=== WEATHER ===
['datetime', 'temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'humidity_pct', 'weather_code', 'date', 'hour', 'day_of_week', 'month', 'year', 'is_precipitation', 'is_cold', 'is_hot', 'feels_like_c']
datetime            datetime64[ns]
temperature_c              fl

In [4]:
# Weather join: match on hourly timestamp
# Weather 'datetime' column = demand 'hour' column (both hourly)

# Select only the weather features we need (avoid duplicate columns)
weather_features = weather[[
    'datetime', 'temperature_c', 'precipitation_mm', 'wind_speed_kmh',
    'humidity_pct', 'weather_code', 'is_precipitation', 'is_cold',
    'is_hot', 'feels_like_c'
]].copy()

# Join on timestamp
demand_weather = demand.merge(
    weather_features,
    left_on='hour',
    right_on='datetime',
    how='left'
)

# Drop the duplicate datetime column
demand_weather = demand_weather.drop(columns=['datetime'])

# Check for missing weather data
weather_nulls = demand_weather['temperature_c'].isnull().sum()
print(f'Rows after weather join: {len(demand_weather):,}')
print(f'Rows missing weather data: {weather_nulls:,} ({weather_nulls/len(demand_weather)*100:.2f}%)')

Rows after weather join: 8,215,056
Rows missing weather data: 0 (0.00%)


In [5]:
# Station join: match station IDs
# demand has 'start_station_id', stations has 'station_id'

# Select useful station features (drop fetched_at, rental_methods, has_kiosk)
station_features = stations[[
    'station_id', 'station_name', 'lat', 'lon', 'capacity'
]].copy()

demand_weather_stations = demand_weather.merge(
    station_features,
    left_on='start_station_id',
    right_on='station_id',
    how='left'
)

# Drop duplicate station_id column
demand_weather_stations = demand_weather_stations.drop(columns=['station_id'])

# Check for unmatched stations
station_nulls = demand_weather_stations['capacity'].isnull().sum()
unmatched = demand_weather_stations[demand_weather_stations['capacity'].isnull()]['start_station_id'].nunique()
print(f'Rows after station join: {len(demand_weather_stations):,}')
print(f'Rows missing station data: {station_nulls:,}')
print(f'Unmatched stations: {unmatched} out of {demand_weather_stations["start_station_id"].nunique()}')

Rows after station join: 8,215,056
Rows missing station data: 8,215,056
Unmatched stations: 534 out of 534


In [6]:
# Look at sample station IDs from both sources
print('Demand station IDs (sample):')
print(demand_weather['start_station_id'].unique()[:10])

print('\nStation metadata IDs (sample):')
print(stations['station_id'].unique()[:10])

Demand station IDs (sample):
['A32000' 'A32001' 'A32002' 'A32003' 'A32004' 'A32005' 'A32006' 'A32008'
 'A32009' 'A32010']

Station metadata IDs (sample):
['2113559364250874028' '5d17f129-af8c-4f30-bc58-6a8279052772'
 'a6d6d4b2-3666-43ad-a690-7f676bdcc3fd'
 'f834e75d-0de8-11e7-991c-3863bb43a7d0'
 '829fa5a8-cc88-4ab2-9f34-b481aaa1612e'
 'f8347ded-0de8-11e7-991c-3863bb43a7d0'
 'eed9e64f-f09d-43c1-aa31-da968c66750a'
 'f8348719-0de8-11e7-991c-3863bb43a7d0'
 '8e7c1984-dc81-43b0-a0d3-58c6de45225a'
 '6eaa7343-402d-41c9-9aac-8c664fdbdcf1']


In [7]:
# Reload a small sample of the original trip data to see all columns
sample = pd.read_parquet(
    f'gs://{BUCKET}/processed/cleaned/year=2024/',
    storage_options=STORAGE_OPTS,
    columns=['start_station_id', 'start_station_name', 'start_lat', 'start_lng']
)

print('Sample from trip data:')
print(sample[['start_station_id', 'start_station_name', 'start_lat', 'start_lng']].drop_duplicates().head(10))

print(f'\nUnique station IDs in trips: {sample["start_station_id"].nunique()}')
print(f'Unique station names in trips: {sample["start_station_name"].nunique()}')

Sample from trip data:
  start_station_id                                 start_station_name  \
0           M32041                    MIT Pacific St at Purrington St   
1           D32019                                   Chinatown T Stop   
2           A32000                                           Fan Pier   
3           B32037                   Deerfield St at Commonwealth Ave   
4           S32034                                         Perry Park   
5           D32032                                         Silber Way   
6           M32025          Linear Park - Mass. Ave. at Cameron Ave.    
7           M32004                                          Kendall T   
8           M32012  Central Sq Post Office / Cambridge City Hall a...   
9           A32043                       Western Ave at Richardson St   

   start_lat  start_lng  
0  42.359573 -71.101295  
1  42.352151 -71.062563  
2  42.353391 -71.044571  
3  42.349244 -71.097282  
4  42.379334 -71.103419  
5  42.349496 -71.

In [8]:
# Check if station names match between trip data and metadata
trip_names = set(sample['start_station_name'].dropna().unique())
meta_names = set(stations['station_name'].dropna().unique())

# How many match exactly?
exact_match = trip_names.intersection(meta_names)
print(f'Trip station names: {len(trip_names)}')
print(f'Metadata station names: {len(meta_names)}')
print(f'Exact matches: {len(exact_match)}')
print(f'In trips but not metadata: {len(trip_names - meta_names)}')
print(f'In metadata but not trips: {len(meta_names - trip_names)}')

# Show some unmatched examples
print('\nSample unmatched trip names:')
for name in sorted(trip_names - meta_names)[:10]:
    print(f'  Trip: {name}')

Trip station names: 541
Metadata station names: 595
Exact matches: 500
In trips but not metadata: 41
In metadata but not trips: 95

Sample unmatched trip names:
  Trip:  Broadway and Cabot
  Trip: 18 Dorrance Warehouse
  Trip: 515 Somerville Ave (Temp. Winter Location)
  Trip: 700 Commonwealth Ave.
  Trip: Adams St at Lonsdale St
  Trip: B.U. Central - 725 Comm. Ave.
  Trip: BCBS Hingham
  Trip: Broadway at Gerrish Ave
  Trip: Brookline Village - Station Street at MBTA
  Trip: CambridgeSide Galleria - CambridgeSide PL at Land Blvd


In [9]:
# Step 1: Create a mapping from trip data (station_id → name, lat, lon)
# Use the most common lat/lon per station (some trips might have slight GPS drift)
trip_stations = sample.groupby('start_station_id').agg({
    'start_station_name': 'first',
    'start_lat': 'median',
    'start_lng': 'median'
}).reset_index()

print(f'Unique stations in trip data: {len(trip_stations)}')

# Step 2: Match by name first
name_match = trip_stations.merge(
    stations[['station_name', 'capacity', 'lat', 'lon']],
    left_on='start_station_name',
    right_on='station_name',
    how='left'
)

matched = name_match['capacity'].notna().sum()
unmatched = name_match['capacity'].isna().sum()
print(f'Matched by name: {matched}')
print(f'Still unmatched: {unmatched}')

Unique stations in trip data: 529
Matched by name: 497
Still unmatched: 32


In [10]:
from scipy.spatial import cKDTree

# Get unmatched stations
unmatched_stations = name_match[name_match['capacity'].isna()].copy()
print(f'Unmatched stations to fix: {len(unmatched_stations)}')

# Build a KDTree from metadata coordinates for fast nearest-neighbor lookup
meta_coords = stations[['lat', 'lon']].values
tree = cKDTree(meta_coords)

# For each unmatched station, find the nearest metadata station
matches = []
for _, row in unmatched_stations.iterrows():
    trip_coord = [row['start_lat'], row['start_lng']]
    distance, idx = tree.query(trip_coord)
    matched_station = stations.iloc[idx]
    matches.append({
        'start_station_id': row['start_station_id'],
        'trip_name': row['start_station_name'],
        'meta_name': matched_station['station_name'],
        'capacity': matched_station['capacity'],
        'meta_lat': matched_station['lat'],
        'meta_lon': matched_station['lon'],
        'distance_deg': distance
    })

coord_matches = pd.DataFrame(matches)

# Show the matches (distance in degrees — ~0.001 ≈ 100 meters)
print('\nCoordinate matches:')
for _, row in coord_matches.iterrows():
    dist_m = row['distance_deg'] * 111_000  # rough conversion to meters
    flag = '⚠️' if dist_m > 500 else '✅'
    print(f'  {flag} {row["trip_name"][:45]:45s} → {row["meta_name"][:45]:45s} ({dist_m:.0f}m)')

Unmatched stations to fix: 32

Coordinate matches:
  ✅ B.U. Central - 725 Comm. Ave.                 → B.U. Central T Stop - 725 Commonwealth Ave    (0m)
  ✅ Washington St at Egremont Rd                  → Washington St at Corey Rd                     (134m)
  ✅ Western Ave at Richardson St                  → Western Ave at Leo Birmingham Pkwy            (128m)
  ✅ Dudley Square - Bolling Building              → Nubian Square - Bolling Building              (0m)
  ✅ Chestnut Hill Ave. at Ledgemere Road          → Chestnut Hill Ave at Ledgemere Rd             (0m)
  ✅ 700 Commonwealth Ave.                         → Commonwealth Ave at Granby St                 (29m)
  ⚠️ BCBS Hingham                                  → Pope John Paul II Park - Neponset Trail at Ha (19985m)
  ✅ Thetford Ave at Norfolk St                    → Milton Ave at Edson St                        (280m)
  ✅ Mass Ave T Station                            → Mass Ave T Stop                               (0m)
  ✅ Adams 

In [11]:
# Set a distance threshold — reject matches over 500 meters
DISTANCE_THRESHOLD_M = 500

coord_matches['distance_m'] = coord_matches['distance_deg'] * 111_000
good_coord_matches = coord_matches[coord_matches['distance_m'] <= DISTANCE_THRESHOLD_M]
bad_coord_matches = coord_matches[coord_matches['distance_m'] > DISTANCE_THRESHOLD_M]

print(f'Good coordinate matches: {len(good_coord_matches)}')
print(f'Rejected (too far): {len(bad_coord_matches)}')
for _, row in bad_coord_matches.iterrows():
    print(f'  ❌ {row["trip_name"]} ({row["distance_m"]:.0f}m)')

# Build complete station lookup: name matches + good coordinate matches
# From name matches
name_matched = name_match[name_match['capacity'].notna()][
    ['start_station_id', 'capacity']
].copy()

# From coordinate matches
coord_matched = good_coord_matches[['start_station_id', 'capacity']].copy()

# Combine
station_lookup = pd.concat([name_matched, coord_matched], ignore_index=True)
print(f'\nTotal stations with metadata: {len(station_lookup)} out of 534')
print(f'Stations without metadata: {534 - len(station_lookup)}')

Good coordinate matches: 29
Rejected (too far): 3
  ❌ BCBS Hingham (19985m)
  ❌  Broadway and Cabot (636m)
  ❌ Sumner St at Shirley Ave (605m)

Total stations with metadata: 526 out of 534
Stations without metadata: 8


In [12]:
# Join station capacity to demand table
demand_weather_stations = demand_weather.merge(
    station_lookup[['start_station_id', 'capacity']],
    on='start_station_id',
    how='left'
)

# Check results
has_capacity = demand_weather_stations['capacity'].notna().sum()
missing_capacity = demand_weather_stations['capacity'].isna().sum()

print(f'Rows with station capacity: {has_capacity:,}')
print(f'Rows missing capacity: {missing_capacity:,} ({missing_capacity/len(demand_weather_stations)*100:.1f}%)')
print(f'\nTotal rows still: {len(demand_weather_stations):,}')

Rows with station capacity: 8,091,984
Rows missing capacity: 123,072 (1.5%)

Total rows still: 8,215,056


In [13]:
# Fill missing capacity with median capacity across all stations
median_capacity = station_lookup['capacity'].median()
print(f'Median station capacity: {median_capacity}')
demand_weather_stations['capacity'] = demand_weather_stations['capacity'].fillna(median_capacity).astype(int)
print(f'Missing capacity after fill: {demand_weather_stations["capacity"].isnull().sum()}')

# Now join holidays
# Demand 'date' is string, holidays 'date' is datetime — need to align
demand_weather_stations['date'] = pd.to_datetime(demand_weather_stations['date'])

# Select only what we need from holidays
holiday_features = holidays[['date', 'is_holiday']].copy()

demand_all = demand_weather_stations.merge(
    holiday_features,
    on='date',
    how='left'
)

# Non-holiday dates won't match — fill with 0
demand_all['is_holiday'] = demand_all['is_holiday'].fillna(0).astype(int)

print(f'\nFinal joined table: {len(demand_all):,} rows')
print(f'Holiday rows: {(demand_all["is_holiday"] == 1).sum():,}')
print(f'Non-holiday rows: {(demand_all["is_holiday"] == 0).sum():,}')
print(f'\nColumns: {demand_all.columns.tolist()}')

Median station capacity: 17.0
Missing capacity after fill: 0

Final joined table: 8,215,056 rows
Holiday rows: 269,136
Non-holiday rows: 7,945,920

Columns: ['start_station_id', 'hour', 'demand_count', 'hour_of_day', 'date', 'year', 'month', 'day_of_week', 'is_weekend', 'temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'humidity_pct', 'weather_code', 'is_precipitation', 'is_cold', 'is_hot', 'feels_like_c', 'capacity', 'is_holiday']


In [14]:
# Sort by station and time (critical for lag calculations)
demand_all = demand_all.sort_values(['start_station_id', 'hour']).reset_index(drop=True)

# Create lag features per station
# demand_lag_1h: demand 1 hour ago
# demand_lag_24h: demand same hour yesterday
# demand_lag_168h: demand same hour last week

print('Creating lag features (this may take a minute)...')

demand_all['demand_lag_1h'] = demand_all.groupby('start_station_id')['demand_count'].shift(1)
print('  ✓ lag 1h')

demand_all['demand_lag_24h'] = demand_all.groupby('start_station_id')['demand_count'].shift(24)
print('  ✓ lag 24h')

demand_all['demand_lag_168h'] = demand_all.groupby('start_station_id')['demand_count'].shift(168)
print('  ✓ lag 168h (1 week)')

# Check nulls — first rows of each station won't have lag values
for col_name in ['demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h']:
    nulls = demand_all[col_name].isnull().sum()
    print(f'  {col_name} nulls: {nulls:,}')

Creating lag features (this may take a minute)...
  ✓ lag 1h
  ✓ lag 24h
  ✓ lag 168h (1 week)
  demand_lag_1h nulls: 534
  demand_lag_24h nulls: 12,816
  demand_lag_168h nulls: 89,712


In [ ]:
# Rolling averages capture smoothed demand trends
# rolling_avg_3h: average demand over past 3 hours
# rolling_avg_6h: average demand over past 6 hours
# rolling_avg_24h: average demand over past 24 hours

print('Creating rolling averages...')

demand_all['rolling_avg_3h'] = demand_all.groupby('start_station_id')['demand_count'] \
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
print('  ✓ rolling 3h')

demand_all['rolling_avg_6h'] = demand_all.groupby('start_station_id')['demand_count'] \
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
print('  ✓ rolling 6h')

demand_all['rolling_avg_24h'] = demand_all.groupby('start_station_id')['demand_count'] \
    .transform(lambda x: x.shift(1).rolling(24, min_periods=1).mean())
print('  ✓ rolling 24h')

# Check nulls
for col_name in ['rolling_avg_3h', 'rolling_avg_6h', 'rolling_avg_24h']:
    nulls = demand_all[col_name].isnull().sum()
    print(f'  {col_name} nulls: {nulls:,}')

Creating rolling averages...
  ✓ rolling 3h
  ✓ rolling 6h
  ✓ rolling 24h
  rolling_avg_3h nulls: 534
  rolling_avg_6h nulls: 534
  rolling_avg_24h nulls: 534


In [16]:
# Fill lag nulls with 0 (first hours of each station — no history available)
lag_cols = ['demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h',
            'rolling_avg_3h', 'rolling_avg_6h', 'rolling_avg_24h']
demand_all[lag_cols] = demand_all[lag_cols].fillna(0)

print(f'Total nulls remaining: {demand_all[lag_cols].isnull().sum().sum()}')

# Cyclical time encoding
# Why? Hour 23 and hour 0 are close in reality but far apart numerically.
# Sin/cos encoding preserves this circular relationship.

demand_all['hour_sin'] = np.sin(2 * np.pi * demand_all['hour_of_day'] / 24)
demand_all['hour_cos'] = np.cos(2 * np.pi * demand_all['hour_of_day'] / 24)
demand_all['dow_sin'] = np.sin(2 * np.pi * demand_all['day_of_week'] / 7)
demand_all['dow_cos'] = np.cos(2 * np.pi * demand_all['day_of_week'] / 7)
demand_all['month_sin'] = np.sin(2 * np.pi * demand_all['month'] / 12)
demand_all['month_cos'] = np.cos(2 * np.pi * demand_all['month'] / 12)

print('Cyclical features added: hour_sin, hour_cos, dow_sin, dow_cos, month_sin, month_cos')
print(f'\nFinal column count: {len(demand_all.columns)}')
print(f'Final row count: {len(demand_all):,}')
print(f'Total nulls in entire table: {demand_all.isnull().sum().sum()}')

Total nulls remaining: 0
Cyclical features added: hour_sin, hour_cos, dow_sin, dow_cos, month_sin, month_cos

Final column count: 32
Final row count: 8,215,056
Total nulls in entire table: 0


In [17]:
print('='*60)
print('FEATURE MATRIX SUMMARY')
print('='*60)

# Group features by category
target = ['demand_count']
time_features = ['hour_of_day', 'day_of_week', 'month', 'year', 'is_weekend',
                 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
weather_features = ['temperature_c', 'precipitation_mm', 'wind_speed_kmh',
                    'humidity_pct', 'feels_like_c', 'is_precipitation', 'is_cold', 'is_hot']
station_features = ['capacity']
context_features = ['is_holiday']
lag_features = ['demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h',
                'rolling_avg_3h', 'rolling_avg_6h', 'rolling_avg_24h']
identifiers = ['start_station_id', 'hour', 'date']

print(f'\nTarget: {target}')
print(f'Time features ({len(time_features)}): {time_features}')
print(f'Weather features ({len(weather_features)}): {weather_features}')
print(f'Station features ({len(station_features)}): {station_features}')
print(f'Context features ({len(context_features)}): {context_features}')
print(f'Lag features ({len(lag_features)}): {lag_features}')
print(f'Identifiers ({len(identifiers)}): {identifiers}')

total_ml_features = len(time_features) + len(weather_features) + len(station_features) + len(context_features) + len(lag_features)
print(f'\nTotal ML features: {total_ml_features}')
print(f'Total rows: {len(demand_all):,}')
print(f'Total nulls: {demand_all.isnull().sum().sum()}')

# Quick check on target distribution
print(f'\nTarget (demand_count) stats:')
print(demand_all['demand_count'].describe())

FEATURE MATRIX SUMMARY

Target: ['demand_count']
Time features (11): ['hour_of_day', 'day_of_week', 'month', 'year', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
Weather features (8): ['temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'humidity_pct', 'feels_like_c', 'is_precipitation', 'is_cold', 'is_hot']
Station features (1): ['capacity']
Context features (1): ['is_holiday']
Lag features (6): ['demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h', 'rolling_avg_3h', 'rolling_avg_6h', 'rolling_avg_24h']
Identifiers (3): ['start_station_id', 'hour', 'date']

Total ML features: 27
Total rows: 8,215,056
Total nulls: 0

Target (demand_count) stats:
count    8.215056e+06
mean     9.594558e-01
std      2.339871e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      1.350000e+02
Name: demand_count, dtype: float64


In [18]:
import os

# Sort for clean output
demand_all = demand_all.sort_values(['start_station_id', 'hour']).reset_index(drop=True)

# Save locally
os.makedirs('data/features', exist_ok=True)
local_path = 'data/features/feature_matrix.parquet'
demand_all.to_parquet(local_path, index=False)
print(f'Saved locally: {local_path}')
print(f'  Size: {os.path.getsize(local_path) / (1024*1024):.1f} MB')

# Upload to GCS
gcs_path = f'gs://{BUCKET}/processed/features/feature_matrix.parquet'
demand_all.to_parquet(
    gcs_path,
    index=False,
    storage_options=STORAGE_OPTS
)
print(f'Uploaded to GCS: {gcs_path}')

Saved locally: data/features/feature_matrix.parquet
  Size: 74.8 MB
Uploaded to GCS: gs://bluebikes-demand-predictor-data/processed/features/feature_matrix.parquet


In [19]:
print('='*60)
print('NOTEBOOK 06 - FEATURE ENGINEERING: COMPLETE')
print('='*60)
print(f'Input:  4 data sources')
print(f'  - Hourly demand: 8,215,056 rows')
print(f'  - Weather: 15,384 hourly records')
print(f'  - Stations: 595 records (526 matched, 8 filled with median)')
print(f'  - Holidays: 24 dates')
print(f'\nOutput: {len(demand_all):,} rows × {len(demand_all.columns)} columns')
print(f'  - 27 ML features + 1 target + 3 identifiers')
print(f'  - Zero nulls')
print(f'\nKey decisions:')
print(f'  - Station ID mismatch: matched by name (497) + coordinates (29)')
print(f'  - 8 unmatched stations: filled capacity with median ({int(median_capacity)})')
print(f'  - Lag features shifted to prevent target leakage')
print(f'  - Cyclical encoding for hour, day of week, month')
print(f'\nSaved:')
print(f'  Local: data/features/feature_matrix.parquet (74.8 MB)')
print(f'  GCS: gs://{BUCKET}/processed/features/feature_matrix.parquet')
print(f'\n→ Next: Notebook 07 - Train/Test Split & Baseline Model')

NOTEBOOK 06 - FEATURE ENGINEERING: COMPLETE
Input:  4 data sources
  - Hourly demand: 8,215,056 rows
  - Weather: 15,384 hourly records
  - Stations: 595 records (526 matched, 8 filled with median)
  - Holidays: 24 dates

Output: 8,215,056 rows × 32 columns
  - 27 ML features + 1 target + 3 identifiers
  - Zero nulls

Key decisions:
  - Station ID mismatch: matched by name (497) + coordinates (29)
  - 8 unmatched stations: filled capacity with median (17)
  - Lag features shifted to prevent target leakage
  - Cyclical encoding for hour, day of week, month

Saved:
  Local: data/features/feature_matrix.parquet (74.8 MB)
  GCS: gs://bluebikes-demand-predictor-data/processed/features/feature_matrix.parquet

→ Next: Notebook 07 - Train/Test Split & Baseline Model
